# EfficientNet training algorithm – 1500 epochs

- This notebook trains EfficientNet-B1 for 1500 epochs, the configuration that yielded the best performance.
- To train a different architecture, change [Model Type] from 'efficientnet_b1' to resnet18, resnet50, or 'densenet169' (note that ResNet models do not use quotation marks).
- To change the number of training epochs, modify the value passed as an argument to [fit_resnet(1500)] (currently set to 1500).
- This notebook logs training metrics to Comet ML. To run it without Comet ML logging, remove or comment out the block defining [Experiment(...)] and all related lines.

## Start

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:64'

In [8]:
from comet_ml import Experiment
from fastai.callback.comet import CometCallback


from fastai.vision.all import *
from fastai.basics import *
from fastai.callback.all import *
from fastai.medical.imaging import *
from fastai.data.transforms import IndexSplitter 
from fastai.data.core import DataLoaders
from fastai.vision.data import PILImage
from PIL import Image   
#import pydicom
from fastai.learner import Learner
from fastai.losses import BCEWithLogitsLossFlat
# from fastai.callback.progress import ProgressCallback
# from fastai.callback.comet import CometCallback

import torch
#import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import train_test_split 
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, precision_score, f1_score
import requests
import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


from fastai.vision.augment import RandTransform
from fastai.vision.core import TensorImage
from torchvision.transforms.functional import rotate as tv_rotate
from torchvision.transforms.functional import affine as tv_affine
#from fastai.vision.augment import brightness
from torchvision.transforms.functional import adjust_brightness
from torchvision.transforms.functional import adjust_contrast
import random
import torch
import math


import random, numpy as np, torch



In [ ]:
# Check if a GPU is available, identify it, and select "device 0"

if torch.cuda.is_available():
    device = torch.device('cuda:1')
    print(f"GPU available: {torch.cuda.get_device_name(1)}")
else:
    device = torch.device('cpu')
    print("GPU not available, using CPU.") 

In [ ]:
# Configure fastai to use the specific device
#hide
from fastai.vision.all import *
defaults.device = device
device


##### Starting cometML experiment

In [ ]:
experiment = Experiment(
    api_key="7...a",
    project_name="_Ef_",
    workspace="experiment_name",
    auto_param_logging=False,  
    auto_metric_logging=False,  
   
)
experiment.set_name('experiment_name') 

In [ ]:

# Path to the folder containing folders B, M, and N, which hold the respective images for each class.
source = Path('/home2/rosmeri/Fastai_Kaggle_Desarrollando/2daBBDD_Lung-Cancer_/lung_cancer_dataset')

#Path to the respective folders for each class.
source_B = Path('/home2/rosmeri/Fastai_Kaggle_Desarrollando/2daBBDD_Lung-Cancer_/lung_cancer_dataset/Bengin')
source_M = Path('/home2/rosmeri/Fastai_Kaggle_Desarrollando/2daBBDD_Lung-Cancer_/lung_cancer_dataset/Malignant')
source_N = Path('/home2/rosmeri/Fastai_Kaggle_Desarrollando/2daBBDD_Lung-Cancer_/lung_cancer_dataset/Normal')

## Data augmentation transformations

In [ ]:

#..................... GAUSSIAN NOISE .......................#

class AddGaussianNoise(RandTransform):

    def __init__(self, std: float=10.0, p: float=0.5):  
        super().__init__(p=p)
        self.std = std
        self.applied = False

    def before_call(self, b, split_idx):
        self.applied = random.random() < self.p

    def encodes(self, img: TensorImage):
        if not self.applied:
            return img
       
        img = img.float() # Convert to float to avoid errors with Byte
        noise = torch.randn_like(img) * self.std
        noisy_img = img + noise
        return noisy_img.clamp(0, 255)  # Clamps to prevent overflow if 8-bit.


#......................... TRANSLATION .....................#

class AddTranslation(RandTransform):

    def __init__(self, max_shift: int = 77, p: float = 0.75):   # 77
        super().__init__(p=p)
        self.max_shift = max_shift
        self.shift_x = 0
        self.shift_y = 0
        self.applied = False # Initially not applied

    def before_call(self, b, split_idx):
        self.applied = random.random() < self.p
        if self.applied:
            self.shift_x = random.randint(-self.max_shift, self.max_shift)
            self.shift_y = random.randint(-self.max_shift, self.max_shift)
 
        else:
            self.shift_x = 0
            self.shift_y = 0

    def encodes(self, img: TensorImage):
        if not self.applied:
            return img
        img = img.float()

        return tv_affine(img, angle=0.0, translate=[self.shift_x, self.shift_y], scale=1.0, shear=[0.0, 0.0])



In [ ]:
#...................... ROTATE ....................#

class MyRandomRotate(RandTransform):
    
    def __init__(self, max_deg=10, p=0.75):  
        super().__init__(p=p)
        self.max_deg = max_deg
        self.last_angle = None  

    def before_call(self, b, split_idx):
        self.last_angle = random.uniform(-self.max_deg, self.max_deg)

    def encodes(self, x: TensorImage):
        return tv_rotate(x, self.last_angle)
        

#...................... ZOOM ....................#

class MyRandomZoom(RandTransform):
   
    def __init__(self, min_zoom=0.9, max_zoom=1.10, p=0.75):
        super().__init__(p=p)
        self.min_zoom = min_zoom
        self.max_zoom = max_zoom
        self.last_zoom = None

    def before_call(self, b, split_idx):
        self.last_zoom = random.uniform(self.min_zoom, self.max_zoom)
        super().before_call(b, split_idx)

    def encodes(self, x: TensorImage):
        if x.ndim == 3:
            # A single image: (C, H, W)
            _, h, w = x.shape
        elif x.ndim == 4:
            # Batch of images: (B, C, H, W)
            _, _, h, w = x.shape
        else:
            raise ValueError(f"Unexpected shape: {x.shape}")

        scale = self.last_zoom
        return tv_affine(x, angle=0.0, translate=[0, 0], scale=scale, shear=[0.0, 0.0])



#......................  FLIP HORIZONTALLY. ...................#

class MyRandomFlip(RandTransform):
    
    def __init__(self, p=0.75):
        super().__init__(p=p)
        self.flipped_h = None

    def before_call(self, b, split_idx):
        self.flipped_h = random.random() < self.p

    def encodes(self, x: TensorImage):
        if self.flipped_h:
            return torch.flip(x, dims=[2])  
        return x

#......................  FLIP VERTICALLY ....................#

class MyRandomFlipVert(RandTransform):
    
    def __init__(self, p=0.75):
        super().__init__(p=p)
        self.flipped_v = None

    def before_call(self, b, split_idx):
        self.flipped_v = random.random() < self.p

    def encodes(self, x: TensorImage):
        if self.flipped_v:
            return torch.flip(x, dims=[1])  
        return x
        

#......................  LIGHTING (BRIGHTNESS AND CONTRAST)   ....................#

class MyRandomLighting(RandTransform):
    
    def __init__(self, max_lighting=0.2, p=0.75):
        super().__init__(p=p)
        self.max_lighting = max_lighting
        self.last_brightness = 1.0
        self.applied = False

    def before_call(self, b, split_idx):
        self.applied = random.random() < self.p
        
        if self.applied:
            '''
            You should do it like this:
            -0.5 * (1 - max_lighting)   and   +0.5 * (1 + max_lighting)
             and then convert it into a valid factor by adding 1.
            '''
            min_val = -0.5 * (1 - self.max_lighting)
            max_val = +0.5 * (1 + self.max_lighting)
            brightness_change = random.uniform(min_val, max_val)
            self.last_brightness = 1 + brightness_change
        else:
            self.last_brightness = 1.0
    
    def encodes(self, x: TensorImage):
        if self.applied:
            return adjust_brightness(x, brightness_factor=self.last_brightness)
        return x


class MyRandomContrast(RandTransform):
    def __init__(self, max_contrast=0.2, p=0.75):
        super().__init__(p=p)
        self.max_contrast = max_contrast
        self.last_contrast = 1.0
        self.applied = False

    def before_call(self, b, split_idx):
        self.applied = random.random() < self.p
        
        if self.applied:
            low = 1 - self.max_contrast
            high = 1 / (1 - self.max_contrast)
            log_low = math.log(low)
            log_high = math.log(high)
            self.last_contrast = math.exp(random.uniform(log_low, log_high))
        else:
            self.last_contrast = 1.0
    
    def encodes(self, x: TensorImage):
        if self.applied:
            return adjust_contrast(x, contrast_factor=self.last_contrast)
        return x


In [ ]:

my_transforms = [
    #ToFloatAndScale(),         
    MyRandomRotate(max_deg=5, p=0.5),
    MyRandomZoom(min_zoom=0.90, max_zoom=1.05, p=0.5),
    #MyRandomFlip(p=0.5), 
    #MyRandomFlipVert(p=0.5),
    MyRandomLighting(max_lighting=0.1, p=0.5),
    MyRandomContrast(max_contrast=0.1, p=0.5),
    AddGaussianNoise(std=5.0, p=0.5),
    AddTranslation(max_shift=20, p=0.5),  #77
    #Clamp(),                   
    Normalize.from_stats(*imagenet_stats),  # *imagenet_stats ### 
]


## Creando el Directorio de los datos

In [ ]:

data_dir = source

dls = ImageDataLoaders.from_folder(
    data_dir,
    valid_pct=0.2,  
    item_tfms=Resize(512, method='pad',pad_mode='zeros' ),  # https://docs.fast.ai/vision.augment.html#resize
    batch_tfms=my_transforms,  
    seed=42,  
    #img_cls=PILImageBW, 
    bs=8, 
    drop_last=False,   
    num_workers=0,
)


In [ ]:
x, y = dls.train.one_batch()
print(f"Input shape: {x.shape}")
print(f"Target shape: {y.shape}")

In [ ]:
#xb, yb = dls.one_batch()
#print(xb.min(), xb.max())  
#show_images((xb * 0.5) + 0.5)  
#show_images(dls.one_batch()[0])
#xb, yb = dls.one_batch()
#show_images(xb, nrows=2, figsize=(12,6))

#Visualización de un batch de imágenes
#dls.show_batch(max_n=15, nrows=5)

In [ ]:
'''
# Get the number of images in each set
num_train = len(dls.train_ds)  
num_valid = len(dls.valid_ds)

# Obtain the class distribution
from collections import Counter

# Extract the classes from the images in the training set
train_labels = [dls.train_ds.items[i].parent.name for i in range(num_train)]
class_distribution = Counter(train_labels)

# Extract the classes from the images in the validation set
valid_labels = [dls.valid_ds.items[i].parent.name for i in range(num_valid)]
class_distribution_valid = Counter(valid_labels)

print(f"Number of images in the training set: {num_train}")
print("Class distribution:")
for class_name, count in class_distribution.items():
    percentage = (count / num_train) * 100  
    print(f"     {class_name}: {count} ({percentage:.2f}%)")  
   
    #print(f"Batch Size: {dls.bs}") 
print(f"     Number of training batches: {len(dls.train)}")

print(f"Número de imágenes en el conjunto de validación: {num_valid}")
print("Distribución de clases:")
for class_name, count in class_distribution_valid.items():
    percentage = (count / num_valid) * 100  # Calcular el porcentaje
    print(f"     {class_name}: {count} ({percentage:.2f}%)")  # Mostrar el porcentaje con 2 decimales
    # Para imprimir información sobre el DataLoader
    #print(f"Batch Size: {dls.bs}") 
print(f"     Number of batches_train: {len(dls.valid)}")
'''

In [7]:
### To see images, comment at random. This is all good.

'''
import comet_ml
from comet_ml import Experiment
from PIL import Image
import numpy as np
import torch


# Obtenemos un batch de imágenes del DataLoader
batch = dls.one_batch()
images, labels = batch

# Ya que las clases se corresponden con un vocabulario de etiquetas
vocab = dls.vocab  # Aquí se contiene el nombre de las clases

# Bucle para cargar algunas imágenes y subirlas a Comet
for i in range(min(15, len(images))):  # Subimos solo 6 imágenes como ejemplo
    img = images[i].cpu().numpy()  # Convertir tensor a numpy array

    # Las imágenes en escala de grises tienen un solo canal
    if img.shape[0] == 1:  # Escala de grises
        img = img.squeeze(0)  # Eliminamos el canal extra
        img = (img * 255).astype(np.uint8)  # Escalar a [0, 255]
        # Convertir la imagen numpy en formato PIL en escala de grises
        image_pil = Image.fromarray(img, mode='L')  # 'L' es el modo para imágenes en escala de grises
    else:  # es que he mantenido las imágenes tal cual está, con 3 canales.
        img = (img * 255).astype(np.uint8)  # Escalar a [0, 255]
        img = img.transpose(1, 2, 0)  # Cambiar a formato (altura, ancho, canales)
        # Convertir la imagen numpy en formato PIL
        image_pil = Image.fromarray(img)  # Modo por defecto es RGB

    # Obtener el nombre de la clase correspondiente a la etiqueta
    class_name = vocab[labels[i].item()]   # Obtener el nombre de las transformaciones aplicadas
    transformations = str(dls.after_batch)  # Esto incluye todas las transformaciones aplicadas

    # Registrar la imagen en Comet
    experiment.log_image(
        image_data=np.array(image_pil),  # Convertir la imagen PIL a numpy array
        name=f"Image - {class_name}",
        image_scale=0.5,
        image_format="png",
        metadata={"label": class_name, "transformation": transformations}
    )
'''

'\nimport comet_ml\nfrom comet_ml import Experiment\nfrom PIL import Image\nimport numpy as np\nimport torch\n\n\n# Obtenemos un batch de imágenes del DataLoader\nbatch = dls.one_batch()\nimages, labels = batch\n\n# Ya que las clases se corresponden con un vocabulario de etiquetas\nvocab = dls.vocab  # Aquí se contiene el nombre de las clases\n\n# Bucle para cargar algunas imágenes y subirlas a Comet\nfor i in range(min(15, len(images))):  # Subimos solo 6 imágenes como ejemplo\n    img = images[i].cpu().numpy()  # Convertir tensor a numpy array\n\n    # Las imágenes en escala de grises tienen un solo canal\n    if img.shape[0] == 1:  # Escala de grises\n        img = img.squeeze(0)  # Eliminamos el canal extra\n        img = (img * 255).astype(np.uint8)  # Escalar a [0, 255]\n        # Convertir la imagen numpy en formato PIL en escala de grises\n        image_pil = Image.fromarray(img, mode=\'L\')  # \'L\' es el modo para imágenes en escala de grises\n    else:  # es que he mantenido

### Training

In [ ]:
'''
Function to delete files with the .pth extension from a specified directory. 
'''
def delete_temp_models(path):
    for file in os.listdir(path):
        if file.endswith(".pth"):
            os.remove(os.path.join(path, file))

 Crear un callback personalizado que
 
- Al final de cada epoch (after_epoch), obtiene un batch del dls.train.

- Convierte las imágenes (ya transformadas) a formato visual.

- Las muestra como salida y en un futuro las puede subir a Comet usando experiment.log_image(). Pero puede q le haga invertir más tiempo a nuestro algoritmo.

In [ ]:
'''
Create a custom callback that:
- At the end of each epoch (`after_epoch`), retrieves a batch from `dls.train`.
- Converts the (already transformed) images into a visual format.
- Displays them as output and could potentially upload them to Comet 
   in the future using `experiment.log_image()`. 
   However, this might increase the processing time for our algorithm.
'''


class DA_per_epoch_Callback(Callback):
    def __init__(self, dls, image_idx=0, figsize=(6, 6)):
        self.dls = dls
        self.image_idx = image_idx
        self.figsize = figsize

    def before_fit(self):
        self.epoch_images = []  # clean before every workout
        # Capture the still image from the original dataset (without transformations yet)
        self.fixed_batch = self.dls.train_ds[:max(4, self.image_idx + 1)]
        self.fixed_xb, self.fixed_yb = zip(*self.fixed_batch)
        self.fixed_xb = torch.stack([self.dls.train.after_item(x) for x in self.fixed_xb])
        self.fixed_yb = tensor(self.fixed_yb)

    def after_epoch(self):

        # This first part ensures the callback doesn't execute during lr_find, avoiding that extra transformation.
        if not hasattr(self.learn, 'training') or len(self.learn.recorder.losses) < 2:
            return  

        if globals().get('in_lr_find', False):
            return
            
        # Extract selected image
        xb = self.fixed_xb[self.image_idx].unsqueeze(0).to(self.dls.device)
        yb = self.fixed_yb[self.image_idx].unsqueeze(0)

        # Apply after_batch transformations
        xb_aug = self.dls.after_batch(xb)

        # Decode (remove normalization, etc.)
        decoded_img, decoded_lbl = self.dls.decode((xb_aug, yb))

        # Show custom transformation parameters (if any)
        print(f"\n [Epoch {self.epoch + 1}] Applied transformations:")
        
        for tfm in self.dls.after_batch.fs:
            if hasattr(tfm, 'last_angle'):
                print(f" - {tfm.__class__.__name__}: applied angle = {tfm.last_angle:.2f}°")
            elif hasattr(tfm, 'last_zoom'):
                print(f" - {tfm.__class__.__name__}: applied zoom = {tfm.last_zoom:.2f}")
            elif hasattr(tfm, 'flipped_h'):
                print(f" - {tfm.__class__.__name__}: flipped_h = {tfm.flipped_h}")
            elif hasattr(tfm, 'flipped_v'):
                print(f" - {tfm.__class__.__name__}: flipped_v = {tfm.flipped_v}")
            elif hasattr(tfm, 'last_brightness'):
                print(f" - {tfm.__class__.__name__}: apllied brightness = {tfm.last_brightness:.2f}")
            elif hasattr(tfm, 'last_contrast'):
                print(f" - {tfm.__class__.__name__}: applied contrast = {tfm.last_contrast:.2f}")
            elif isinstance(tfm, AddGaussianNoise):
                print(f" - {tfm.__class__.__name__}: noise {'aplicado' if tfm.applied else 'no aplicado'}, std = {tfm.std}")
            elif isinstance(tfm, AddTranslation):
                print(f" - {tfm.__class__.__name__}: traslation {'aplicada' if tfm.applied else 'no aplicada'}, shift_x = {tfm.shift_x}, shift_y = {tfm.shift_y}")

       # Display image the way fastai does internally (more natural)
        _, ax = plt.subplots(figsize=(5, 5), dpi=100)  
        
        show_image(decoded_img[0], title=f"Label: {decoded_lbl[0]}", ax=ax)
        

        # these two lines ensure the callback displays grayscale images, since
        # I identified them as such in the dls
        #cmap = 'gray' if decoded_img[0].shape[0] == 1 else None
        #show_image(decoded_img[0], title=f"Label: {decoded_lbl[0]}", ax=ax, cmap=cmap)
        plt.suptitle(f"Epoch {self.epoch + 1}")
        plt.tight_layout()
        plt.show()


In [ ]:
'''
Biblioteca Pytorch Image Models (timm)
'''

import timm

# Mostrar todos los modelos disponibles (más de 1000)
# model_names = timm.list_models()
# print(model_names[:50])  # Muestra los primeros 50

# Listar modelos de timm con un nombre específico
#timm.list_models('*resnetv2*')
#timm.list_models('*efficientnet*')
#timm.list_models('*densenet*')

# Crear un modelo con timm,
#model = timm.create_model('resnetv2_50', pretrained=True, num_classes=dls.c)

# pero gracias a ... ya sé que no hace falta
'''
https://timm.fast.ai/
'''


In [9]:
from fastai.callback.tracker import SaveModelCallback
from io import BytesIO
from fastai.callback.tracker import EarlyStoppingCallback
from fastai.callback.training import GradientClip

In [ ]:

in_lr_find = False

def fit_resnet(num_epochs,lr=None):   #  ,lr=None

    global in_lr_find
    
    learn = vision_learner(
        dls,
        'efficientnet_b1', #'densenet169',  resnet50, resnet18
        pretrained=True,
        # n_in=1,
        loss_func=CrossEntropyLossFlat(),
        metrics=[accuracy, Precision(average='macro'), Recall(average='macro'), F1Score(average='macro')],
        model_dir='_Ef_',
        cbs=[DA_per_epoch_Callback(dls, image_idx=2, figsize=(5,5)), SaveModelCallback(monitor='valid_loss', fname='E-1500')],
    ).to_fp16()
   
     # This isn't strictly necessary, but just in case
     # find the optimal max_lr using lr_find

    
    if lr is None:
        try:
            in_lr_find = True
             
            lr_min, lr_steep, lr_valley, lr_slide = learn.lr_find(suggest_funcs=(minimum, steep, valley, slide))
            
            print("\n Sugerencias de learning rate:")
            print(f" - minimum: {lr_min:.2e}")
            print(f" - steep  : {lr_steep:.2e} ") 
            print(f" - valley : {lr_valley:.2e}")
            print(f" - slide  : {lr_slide:.2e}") 

            # Deciding on the best learning rate
            if lr_valley > 1e-6 and lr_valley < 1e-5:
                #  if lr_valley > 1e-5 and lr_valley < 1e-4:
                lr_max = lr_valley
                print(f"\n Using valley: {lr_valley:.2e}")
            else:
                lr_max = 1e-5 #1e-4
                print(f"\n  Very low valley and slide. Using default value.: {lr_max:.2e}")

            
            
        except Exception as e:
            lr_max = 1e-5  #1e-4
            print(f"  Error in lr_find: {e}. Using default value: {lr_max}")
        finally:
            in_lr_find = False
    else:
        lr_max = lr
        print(f" Using a manually provided learning rate: {lr_max:.2e}")

        
    learn.freeze()
    learn.fit_one_cycle(10, lr_valley)
    learn.unfreeze()  

    # Start the training cycle with fit_one_cycle for all epochs
    learn.fit_one_cycle(num_epochs-10, lr_max)
    
    learn.recorder.plot_sched()  # lr PLOT
    plt.show()
    
    # Save the image to an in-memory buffer
    buf = BytesIO()
    plt.savefig(buf, format='png')
    buf.seek(0)

    # Send directly to Comet
    experiment.log_image(buf, name="lr_schedule_plot", image_format="png")

    # Show summary of used LRs
    lrs = learn.recorder.lrs         
    print(f"\n Learning rates per step: {lrs}") #[:10]
    print(f" Maximum LR used in the cycle: {max(lrs):.2e}")

    num_epochs_logged = len(learn.recorder.values)  # number of sublists within recorder.values

    # Iterating only over the recorded values
    for epoch in range(num_epochs_logged):
        epoch_values = learn.recorder.values[epoch] # All the values of the current era
        train_loss = epoch_values[0]  
        valid_loss = epoch_values[1]  
        accuracy_val = epoch_values[2]  
        precision_val = epoch_values[3]  
        recall_val = epoch_values[4]  
        f1_val = epoch_values[5]  
        
        # Logging loss metrics to Comet
        experiment.log_metric("train_loss_ros", train_loss, step=epoch+1)
        experiment.log_metric("valid_loss_ros", valid_loss, step=epoch+1)
        experiment.log_metric("accuracy", accuracy_val, step=epoch+1)
        experiment.log_metric("precision", precision_val, step=epoch+1)
        experiment.log_metric("recall", recall_val, step=epoch+1)
        experiment.log_metric("f1_score", f1_val, step=epoch+1)

        delete_temp_models('/tmp')
   
    return learn
    

In [ ]:
learn_finish=fit_resnet(1500)  # , lr=None

In [ ]:
learn_finish.summary()

In [ ]:
torch.cuda.empty_cache()

# Finalizar el experimento
experiment.end()
